In [1]:
import pandas as pd
orders = pd.read_csv("orders.csv")
orders.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [2]:
users = pd.read_json("users.json")
users.head()

,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [5]:
final = orders.merge(users,on="user_id",how="left")
final.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular


In [8]:
gold_df=final[final['membership']=='Gold']


In [9]:
city_revenue=gold_df.groupby('city')['total_amount'].sum()

In [10]:
highest_city=city_revenue.idxmax()
highest_amount=city_revenue.max()

print("city with highest revenue from gold:",highest_city)
print("total:",highest_amount)

city with highesr revenue from gold: Chennai
total: 1080909.79


In [11]:
final.columns

Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name', 'name', 'city', 'membership'],
      dtype='object')

In [15]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")


In [16]:

with open("restaurants.sql", "r") as f:
    sql_script = f.read()

conn.executescript(sql_script)


In [17]:
restaurants = pd.read_sql("SELECT * FROM restaurants", conn)
restaurants.head()


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [65]:
final_orders = final.merge(restaurants, left_on='restaurant_id', right_on='restaurant_id', how='left')


In [21]:
cuisine_avg = final_orders.groupby('cuisine')['total_amount'].mean()
highest_cuisine = cuisine_avg.idxmax()
highest_value = cuisine_avg.max()

print("Cuisine with highest average order value:", highest_cuisine)
print("Average order value:", highest_value)


Cuisine with highest average order value: Mexican
Average order value: 808.0213444401395


In [22]:
user_total = final.groupby('user_id')['total_amount'].sum()


In [23]:
big_spenders = user_total[user_total > 1000]


In [24]:
num_users = big_spenders.shape[0]
print("Number of distinct users with total orders > ₹1000:", num_users)


Number of distinct users with total orders > ₹1000: 2544


In [25]:
bins = [2.9, 3.5, 4.0, 4.5, 5.0]
labels = ['3.0-3.5', '3.6-4.0', '4.1-4.5', '4.6-5.0']

final_orders['rating_range'] = pd.cut(final_orders['rating'], bins=bins, labels=labels, right=True)


In [27]:
rating_revenue = final_orders.groupby('rating_range')['total_amount'].sum()
highest_range = rating_revenue.idxmax()
highest_value = rating_revenue.max()

print("Rating range with highest total revenue:", highest_range)
print("Total revenue:", highest_value)


Rating range with highest total revenue: 4.6-5.0
Total revenue: 2197030.75


/tmp/ipython-input-2797155122.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rating_revenue = final_orders.groupby('rating_range')['total_amount'].sum()


In [28]:
gold_df = final[final['membership'] == 'Gold']


In [29]:
city_avg = gold_df.groupby('city')['total_amount'].mean()


In [30]:
highest_city = city_avg.idxmax()
highest_value = city_avg.max()

print("City with highest average order value among Gold members:", highest_city)
print("Average order value:", highest_value)


City with highest average order value among Gold members: Chennai
Average order value: 808.4590800299178


In [31]:
distinct_res = final_orders.groupby('cuisine')['restaurant_id'].nunique()


In [32]:
cuisine_revenue = final_orders.groupby('cuisine')['total_amount'].sum()


In [33]:
cuisine_summary = pd.DataFrame({
    'num_restaurants': distinct_res,
    'total_revenue': cuisine_revenue
})

cuisine_summary.sort_values(by='num_restaurants', inplace=True)
cuisine_summary


,num_restaurants,total_revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [34]:

target_cuisine = cuisine_summary[cuisine_summary['num_restaurants'] == cuisine_summary['num_restaurants'].min()]
target_cuisine.sort_values(by='total_revenue', ascending=False)


,num_restaurants,total_revenue
cuisine,,
Chinese,120,1930504.65


In [35]:
total_orders = final.shape[0]


In [36]:
gold_orders = final[final['membership'] == 'Gold'].shape[0]


In [37]:
percentage = round((gold_orders / total_orders) * 100)
print("Percentage of total orders by Gold members:", percentage)


Percentage of total orders by Gold members: 50


In [38]:
res_stats = final.groupby(['restaurant_id','restaurant_name']).agg(
    total_orders = ('order_id','count'),
    avg_order_value = ('total_amount','mean')
).reset_index()


In [39]:
low_orders_res = res_stats[res_stats['total_orders'] < 20]


In [41]:
target_res = low_orders_res.loc[low_orders_res['avg_order_value'].idxmax()]

print("Restaurant with highest average order value but <20 orders:")
print(target_res[['restaurant_name','total_orders','avg_order_value']])


Restaurant with highest average order value but <20 orders:
restaurant_name    Hotel Dhaba Multicuisine
total_orders                             13
avg_order_value                 1040.222308
Name: 293, dtype: object


In [42]:

combo_revenue = final_orders.groupby(['membership','cuisine'])['total_amount'].sum()

highest_combo = combo_revenue.idxmax()
highest_value = combo_revenue.max()

print("Combination with highest revenue:", highest_combo)
print("Total revenue:", highest_value)


Combination with highest revenue: ('Regular', 'Mexican')
Total revenue: 1072943.3


In [43]:
options = [('Gold','Indian'), ('Gold','Italian'), ('Regular','Indian'), ('Regular','Chinese')]
combo_revenue_options = combo_revenue[options]
highest_option_combo = combo_revenue_options.idxmax()
print("Pick this option:", highest_option_combo)


Pick this option: ('Gold', 'Italian')


In [44]:
final['order_date'] = pd.to_datetime(final['order_date'])


/tmp/ipython-input-5126196.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  final['order_date'] = pd.to_datetime(final['order_date'])


In [45]:
final['quarter'] = final['order_date'].dt.quarter


In [46]:
quarter_revenue = final.groupby('quarter')['total_amount'].sum()
highest_quarter = quarter_revenue.idxmax()
highest_value = quarter_revenue.max()

print("Quarter with highest total revenue:", highest_quarter)
print("Total revenue:", highest_value)


Quarter with highest total revenue: 3
Total revenue: 2037385.1


In [47]:
gold_orders = final[final['membership'] == 'Gold']


In [48]:
total_gold_orders = gold_orders.shape[0]
print("Total orders by Gold members:", total_gold_orders)


Total orders by Gold members: 4987


In [49]:
hyderabad_orders = final[final['city'] == 'Hyderabad']


In [50]:
total_revenue_hyderabad = round(hyderabad_orders['total_amount'].sum())
print("Total revenue from Hyderabad:", total_revenue_hyderabad)


Total revenue from Hyderabad: 1889367


In [51]:
distinct_users = final['user_id'].nunique()
print("Number of distinct users who placed at least one order:", distinct_users)


Number of distinct users who placed at least one order: 2883


In [52]:
gold_orders = final[final['membership'] == 'Gold']


In [53]:
avg_order_value_gold = round(gold_orders['total_amount'].mean(), 2)
print("Average order value for Gold members:", avg_order_value_gold)


Average order value for Gold members: 797.15


In [54]:
high_rating_orders = final_orders[final_orders['rating'] >= 4.5]



In [55]:
num_high_rating_orders = high_rating_orders.shape[0]
print("Number of orders for restaurants with rating ≥ 4.5:", num_high_rating_orders)


Number of orders for restaurants with rating ≥ 4.5: 3374


In [56]:
gold_orders = final[final['membership'] == 'Gold']


In [57]:
city_revenue = gold_orders.groupby('city')['total_amount'].sum()
top_city = city_revenue.idxmax()


In [58]:
orders_top_city = gold_orders[gold_orders['city'] == top_city].shape[0]
print("Number of orders in top revenue city among Gold members:", orders_top_city)


Number of orders in top revenue city among Gold members: 1337


In [59]:
final.shape[0]


10000

In [61]:
final = orders.merge(users, on='user_id', how='left')

final_orders = final.merge(restaurants, on='restaurant_id', how='left')
final_orders.shape[0]


10000

In [62]:
import pandas as pd

orders = pd.read_csv('orders.csv')

users = pd.read_json('users.json')


In [63]:

final = orders.merge(users, on='user_id', how='left')


In [64]:
no_match_rows = final[final['name'].isna()]
print(no_match_rows.head())
print("Number of unmatched users:", no_match_rows.shape[0])


Empty DataFrame
Columns: [order_id, user_id, restaurant_id, order_date, total_amount, restaurant_name, name, city, membership]
Index: []
Number of unmatched users: 0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')